# OmniBird — Option A: symmetric per-event encoders + patch-level loss

Both encoders are **per-event** OmniBirdEncoders with `grouped_hierarchical`
attention. EMA from context → target works because the architectures are
identical. The "asymmetry" lives in **what each encoder sees**:

| | Input | Cost | Output |
|---|---|---|---|
| **context_encoder** | events in 192 context patches (~6144 events) | grad + fwd  | per-event ctx features |
| **target_encoder**  | all events (~8192)                              | fwd only    | **per-event** target features |
| **predictor**       | 64 patch-centroid queries → 192 ctx patch tokens | perceiver   | per-patch features |

**Aggregation:** per-event features → per-patch via mean-pool over each
patch's K events (with per-event KPM masking). JEPA loss is **patch-level**
(stable, Point-MAE convention) but the encoders still produce fine-grained
per-event features available for downstream tasks.

**Outline**:
1. Setup
2. Dataset
3. **Tokenization visualization** (raw → Hilbert sort → patches → centroids → masking)
4. Models (symmetric per-event)
5. Optim + resume
6. Patch probe
7. Training loop
8. Plots


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba

from omnibird import (
    OmniBirdConfig,
    PatchOmniBirdEncoder, OmniBirdEncoder, PerceiverPredictor,
    OmniBirdPatchDataset,
    TargetCenter, ema_update, make_momentum_schedule,
    jepa_loss, diag_dict, fmt_diag,
    save_atomic, ensure_dir, short_params, count_params,
    precompute_grid_orderings, invert_perm, quantize_coords,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

DATA_ROOT  = "../data/cifar10_dvs_omnibird"
TRAIN_FRAC = 0.8

cfg = OmniBirdConfig()
cfg.coord_dim       = 3
cfg.signal_dim      = 2                # one-hot ON/OFF
cfg.n_events_total  = 0
cfg.n_events_max    = 4096             # P_max * K_per = 128 * 32 (fits target fwd on 4 GPUs)

# Patch mode (for context + dataset)
cfg.patch_mode         = True
cfg.patch_size         = 32
cfg.patches_per_block  = 16
cfg.n_pred_blocks      = 4
cfg.n_tgt              = cfg.n_pred_blocks * cfg.patches_per_block      # 64 target patches
cfg.ctx_max_patches    = 80           # context: ~62% of pool patches
cfg.patch_curve        = "hilbert"
cfg.context_target_margin = 0.0
cfg.side               = 64

# Target encoder uses per-event grouped_hierarchical attention
cfg.attention_type   = "grouped_hierarchical"
cfg.group_size       = 16
cfg.pool             = "mean"
cfg.use_centroid_pos = True

# JEPA
cfg.loss_type      = "cosine"
cfg.use_centering  = False
cfg.ema_start      = 0.999
cfg.ema_end        = 0.9999
cfg.probe_patience = 5

# Optim
cfg.epochs         = 100
cfg.batch_size     = 64
cfg.lr             = 8e-4
cfg.probe_interval = 10
cfg.probe_epochs   = 3
cfg.log_every      = 25
cfg.ckpt_dir       = "./checkpoints_omnibird_cifar10_dvs_optA"
ensure_dir(cfg.ckpt_dir)

print(f"patches_total = {cfg.n_events_max // cfg.patch_size}  K = {cfg.patch_size}")
print(f"context patches = {cfg.ctx_max_patches}  target patches = {cfg.n_tgt}")


## 2. Dataset

In [ ]:
class CIFAR10DVSFromClips:
    def __init__(self, root, sensor_hw=(128, 128)):
        self.root = root; self.h, self.w = sensor_hw
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(f"No clip_* in {root} (resolved: {os.path.abspath(root)})")
    def __len__(self): return len(self.clips)
    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        if ev.shape[0] == 0:
            ev = np.zeros((1, 4), dtype=np.float32)
        x = ev[:, 0] / max(self.w - 1, 1) * 2.0 - 1.0
        y = ev[:, 1] / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = ev[:, 2]
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = ev[:, 3]
        if p.max() <= 1.0 and p.min() >= 0.0: p = p * 2.0 - 1.0
        out = np.stack([x, y, t, p], axis=1).astype(np.float32)
        with open(os.path.join(clip, "label_0.txt")) as f: label = int(f.read().strip())
        return out, label


base = CIFAR10DVSFromClips(DATA_ROOT)
n = len(base); rng = np.random.RandomState(0)
perm = rng.permutation(n); n_train = int(n * TRAIN_FRAC)
train_idx, test_idx = perm[:n_train], perm[n_train:]
class _Subset:
    def __init__(self, b, i): self.b=b; self.i=i
    def __len__(self): return len(self.i)
    def __getitem__(self, k): return self.b[int(self.i[k])]
base_train = _Subset(base, train_idx); base_test = _Subset(base, test_idx)

from torch.utils.data import DataLoader
mk = lambda b, tr: OmniBirdPatchDataset(b, cfg, train=tr,
                                          patches_per_block=cfg.patches_per_block,
                                          n_pred_blocks=cfg.n_pred_blocks,
                                          ctx_max=cfg.ctx_max_patches,
                                          patch_size=cfg.patch_size,
                                          margin=cfg.context_target_margin,
                                          curve=cfg.patch_curve)
train_ds, train_eval_ds, test_ds = mk(base_train, True), mk(base_train, False), mk(base_test, False)
train_loader      = DataLoader(train_ds,      batch_size=cfg.batch_size, shuffle=True,  num_workers=0, pin_memory=False)
train_eval_loader = DataLoader(train_eval_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0, pin_memory=False)
test_loader       = DataLoader(test_ds,       batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False)
print(f"loaded CIFAR10-DVS: train={len(train_ds)}  test={len(test_ds)}")


## 3. Visualization of the tokenization

Drill into ONE sample and show the full pipeline:

1. Raw events (3D scatter, colored by polarity)
2. After Hilbert curve sort (colored by curve rank)
3. Patches (colored by patch index)
4. Patch centroids (overlaid on events)
5. Multi-block masking: target patches (4 colors) + context patches + leftover


In [ ]:
# Pick one sample
SAMPLE_IDX = 0
sample = train_ds[SAMPLE_IDX]
classes = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
print(f"sample idx={SAMPLE_IDX}, label={classes[sample['label']]}")

P  = sample["pool_patch_events"].shape[0]
K  = sample["pool_patch_events"].shape[1]
F  = sample["pool_patch_events"].shape[2]
pool_ev   = sample["pool_patch_events"].numpy()        # (P, K, 3+signal_dim)
centroids = sample["pool_patch_centroids"].numpy()      # (P, 3)
patch_kpm = sample["pool_patch_kpm"].numpy()             # (P,)
ev_kpm    = sample["pool_event_kpm"].numpy()             # (P, K)
ctx_idx   = sample["ctx_patch_idx"].numpy()
tgt_idx   = sample["tgt_patch_idx"].numpy()

# Flat per-event view
events_flat = pool_ev.reshape(P*K, F)                    # (P*K, F)
coords = events_flat[:, :3]
# One-hot polarity → recover ±1 (signal_dim=2 here): col 0 = ON, col 1 = OFF
pol = events_flat[:, 3] - events_flat[:, 4]
ev_kpm_flat = ev_kpm.reshape(P*K)
real_mask = ~ev_kpm_flat
real_coords = coords[real_mask]
real_pol    = pol[real_mask]

print(f"  P (patches) = {P},  K (events/patch) = {K}")
print(f"  real events = {real_mask.sum()},  padded = {(~real_mask).sum()}")
print(f"  context patches: {len(ctx_idx)} (real = {(~patch_kpm[ctx_idx]).sum()})")
print(f"  target patches : {len(tgt_idx)}")


In [ ]:
# Figure 1: raw → Hilbert sorted → patches
fig = plt.figure(figsize=(20, 6))

# (a) Raw events colored by polarity
ax = fig.add_subplot(1, 3, 1, projection='3d')
pos = real_coords[real_pol > 0]; neg = real_coords[real_pol < 0]
ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], s=1, c='red',  alpha=0.4, label='ON (+1)')
ax.scatter(neg[:, 0], neg[:, 1], neg[:, 2], s=1, c='blue', alpha=0.4, label='OFF (-1)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title(f"(a) raw events  ({real_mask.sum()})\\ncolored by polarity")
ax.legend(loc='lower right', fontsize=8)

# (b) Hilbert curve order — color by rank
ax = fig.add_subplot(1, 3, 2, projection='3d')
# Events are ALREADY sorted by Hilbert in the patch dataset, so rank = sequential index
rank = np.arange(P*K)[real_mask]
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=2, c=rank, cmap='turbo', alpha=0.7)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title(f"(b) Hilbert-sorted\\ncolor = curve rank (low → high)")

# (c) Patches — color by patch index
ax = fig.add_subplot(1, 3, 3, projection='3d')
patch_idx_per_event = np.repeat(np.arange(P), K)[real_mask]
# Random palette so adjacent patches are different colors
rng_cmap = np.random.RandomState(0)
patch_colors = plt.cm.tab20(rng_cmap.permutation(P) % 20)
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=3, c=patch_colors[patch_idx_per_event], alpha=0.8)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title(f"(c) patches  ({P} of size K={K})\\ncolor = patch index")
plt.tight_layout(); plt.show()


In [ ]:
# Figure 2: centroids + multi-block masking
fig = plt.figure(figsize=(20, 6))

# (a) Patch centroids overlaid on events (real)
ax = fig.add_subplot(1, 3, 1, projection='3d')
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=1, c='lightgray', alpha=0.3)
real_cen = centroids[~patch_kpm]
ax.scatter(real_cen[:, 0], real_cen[:, 1], real_cen[:, 2],
           s=20, c='black', marker='o', edgecolors='red', linewidths=0.5,
           alpha=0.85, label=f'patch centroids ({(~patch_kpm).sum()})')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title("(a) patch centroids (black dots)\\nover the event cloud (gray)")
ax.legend(loc='lower right', fontsize=8)

# (b) Target patches: 4 colors, each block one color
TGT_COLORS = ['#fbbf24', '#34d399', '#60a5fa', '#f472b6']
ax = fig.add_subplot(1, 3, 2, projection='3d')
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=1, c='lightgray', alpha=0.25)
patches_per_block = cfg.patches_per_block
for k in range(cfg.n_pred_blocks):
    block_patch_ids = tgt_idx[k*patches_per_block:(k+1)*patches_per_block]
    block_events_mask = np.isin(np.repeat(np.arange(P), K)[real_mask], block_patch_ids)
    block_coords = real_coords[block_events_mask]
    ax.scatter(block_coords[:, 0], block_coords[:, 1], block_coords[:, 2],
               s=8, c=TGT_COLORS[k], alpha=0.7,
               label=f'target block {k+1} ({patches_per_block} patches)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title(f"(b) {cfg.n_pred_blocks} target blocks\\n{cfg.n_pred_blocks * patches_per_block} target patches total")
ax.legend(loc='lower right', fontsize=7, ncol=2)

# (c) Context patches vs leftover
ax = fig.add_subplot(1, 3, 3, projection='3d')
in_ctx = np.isin(np.repeat(np.arange(P), K)[real_mask], ctx_idx)
in_tgt = np.isin(np.repeat(np.arange(P), K)[real_mask], tgt_idx)
leftover = ~(in_ctx | in_tgt)
ax.scatter(real_coords[leftover, 0], real_coords[leftover, 1], real_coords[leftover, 2],
           s=1, c='lightgray', alpha=0.2, label=f'leftover ({leftover.sum()})')
ax.scatter(real_coords[in_ctx, 0], real_coords[in_ctx, 1], real_coords[in_ctx, 2],
           s=2, c='#ef4444', alpha=0.5,
           label=f'context ({(~patch_kpm[ctx_idx]).sum()} real patches)')
for k in range(cfg.n_pred_blocks):
    block_patch_ids = tgt_idx[k*patches_per_block:(k+1)*patches_per_block]
    block_events_mask = np.isin(np.repeat(np.arange(P), K)[real_mask], block_patch_ids)
    block_coords = real_coords[block_events_mask]
    ax.scatter(block_coords[:, 0], block_coords[:, 1], block_coords[:, 2],
               s=5, c=TGT_COLORS[k], alpha=0.65)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
ax.set_title("(c) multi-block masking overview\\nred=ctx, colors=4 target blocks, gray=leftover")
ax.legend(loc='lower right', fontsize=7)

plt.tight_layout(); plt.show()


In [ ]:
# Figure 3: zoom into one patch — show the mini-PointNet input
fig = plt.figure(figsize=(18, 5))

# Pick a real, central patch
real_patch_ids = np.where(~patch_kpm)[0]
chosen = real_patch_ids[len(real_patch_ids)//2]
ev_in_patch = pool_ev[chosen]                          # (K, 3+signal_dim)
ev_kpm_in_patch = ev_kpm[chosen]                       # (K,)
real_in_patch = ~ev_kpm_in_patch
pc = ev_in_patch[real_in_patch, :3]
pp = ev_in_patch[real_in_patch, 3] - ev_in_patch[real_in_patch, 4]

# (a) Where this patch sits in the full cloud
ax = fig.add_subplot(1, 3, 1, projection='3d')
ax.scatter(real_coords[:, 0], real_coords[:, 1], real_coords[:, 2],
           s=1, c='lightgray', alpha=0.2)
ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], s=30, c='red', alpha=0.8,
           label=f'patch {chosen} ({len(pc)} events)')
ax.scatter([centroids[chosen, 0]], [centroids[chosen, 1]], [centroids[chosen, 2]],
            s=200, marker='*', c='black', edgecolors='red', linewidths=1.5,
            label='centroid')
ax.set_title(f"(a) patch {chosen} in the cloud")
ax.legend(loc='lower right', fontsize=8)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')

# (b) Patch in absolute coords
ax = fig.add_subplot(1, 3, 2, projection='3d')
pos_p = pc[pp > 0]; neg_p = pc[pp < 0]
ax.scatter(pos_p[:, 0], pos_p[:, 1], pos_p[:, 2], s=40, c='red',  alpha=0.7)
ax.scatter(neg_p[:, 0], neg_p[:, 1], neg_p[:, 2], s=40, c='blue', alpha=0.7)
ax.scatter([centroids[chosen, 0]], [centroids[chosen, 1]], [centroids[chosen, 2]],
            s=250, marker='*', c='black', edgecolors='yellow', linewidths=1.5)
ax.set_title(f"(b) events in patch {chosen} (absolute coords)")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')

# (c) Relative coords (what mini-PointNet sees)
ax = fig.add_subplot(1, 3, 3, projection='3d')
rel = pc - centroids[chosen]
pos_r = rel[pp > 0]; neg_r = rel[pp < 0]
ax.scatter(pos_r[:, 0], pos_r[:, 1], pos_r[:, 2], s=40, c='red',  alpha=0.7)
ax.scatter(neg_r[:, 0], neg_r[:, 1], neg_r[:, 2], s=40, c='blue', alpha=0.7)
ax.scatter([0], [0], [0], s=250, marker='*', c='black', edgecolors='yellow', linewidths=1.5,
            label='(0,0,0) ← centroid')
ax.set_title(f"(c) RELATIVE coords (input to mini-PointNet)\\nrel = coord − centroid")
ax.legend(loc='lower right', fontsize=8)
ax.set_xlabel('Δx'); ax.set_ylabel('Δy'); ax.set_zlabel('Δt')

plt.tight_layout(); plt.show()


## 4. Symmetric per-event encoders

In [ ]:
def make_event_encoder():
    return OmniBirdEncoder(
        d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
        n_heads=cfg.n_heads, dim_head=cfg.dim_head,
        block_size=cfg.block_size, window=cfg.window,
        n_random=cfg.n_random, n_global=cfg.n_global,
        ffn_mult=cfg.ffn_mult,
        signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
        fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
        serial_orders=cfg.serial_orders,
        reinject_pos=cfg.reinject_pos,
        attention_type=cfg.attention_type,    # grouped_hierarchical
        group_size=cfg.group_size,
        pool=cfg.pool,
        use_centroid_pos=cfg.use_centroid_pos,
    )

context_encoder = make_event_encoder().to(DEVICE)
target_encoder  = make_event_encoder().to(DEVICE)
# Init target as a copy of context so EMA starts from a sensible point
target_encoder.load_state_dict(context_encoder.state_dict())
for q in target_encoder.parameters(): q.requires_grad_(False)

predictor = PerceiverPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
).to(DEVICE)
target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)

def _unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m

print(f"context_encoder: {short_params(_unwrap(context_encoder))}")
print(f"target_encoder : {short_params(_unwrap(target_encoder))}")
print(f"predictor      : {short_params(_unwrap(predictor))}")


## 5. Optim + resume + grid_ranks for per-event target orderings

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

# Precomputed grid ranks for computing per-event orderings on the flattened pool
grid_ranks = {k: v.to(DEVICE) for k, v in precompute_grid_orderings(cfg.side, ndim=3).items()}

PBB_LAST       = os.path.join(cfg.ckpt_dir, "omnibird_last.pt")
PBB_BEST       = os.path.join(cfg.ckpt_dir, "omnibird_best.pt")
PBB_PROBE_BEST = os.path.join(cfg.ckpt_dir, "omnibird_probe_best.pt")

RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
best_probe_acc, probe_no_improve = -1.0, 0
m = cfg.ema_start

if RESUME and os.path.exists(PBB_LAST):
    s = torch.load(PBB_LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    best_probe_acc = s.get("best_probe_acc", -1.0); probe_no_improve = s.get("probe_no_improve", 0)
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}")
else:
    print("starting fresh.")


## 6. Helper: orderings for the per-event view

In [ ]:
def event_orderings(coords, kpm=None):
    # coords: (B, N, coord_dim) flat per-event coords; kpm: (B, N) True at padded.
    # Returns dict[name -> {'perm': (B, N) long, 'inverse': (B, N) long}]
    B, N, _ = coords.shape
    cell_ids = quantize_coords(coords, side=cfg.side, value_range=(-1.0, 1.0))
    if kpm is not None:
        sort_key = kpm.long() * (N + 1)              # push pads to tail
    else:
        sort_key = 0
    out = {}
    for name in ('z', 'z_rev', 'hilbert', 'hilbert_rev'):
        ranks = grid_ranks[name][cell_ids]
        eff = ranks + sort_key
        perm = eff.argsort(dim=-1)
        inv = invert_perm(perm)
        out[name] = {"perm": perm, "inverse": inv}
    return out


## 7. Patch probe (mean-pool ctx patch tokens)

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)


def gather_ctx_subset(batch):
    # Gather raw events + centroids + masks for the context patches.
    pool_ev    = batch["pool_patch_events"].to(DEVICE)
    pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
    pool_kpm   = batch["pool_patch_kpm"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    ctx_idx    = batch["ctx_patch_idx"].to(DEVICE)
    B, P, K, Fdim = pool_ev.shape
    Bc, Pc = ctx_idx.shape
    sub_ev    = torch.gather(pool_ev,    1, ctx_idx[..., None, None].expand(Bc, Pc, K, Fdim))
    sub_cen   = torch.gather(pool_cen,   1, ctx_idx[..., None].expand(Bc, Pc, pool_cen.size(-1)))
    sub_kpm   = torch.gather(pool_kpm,   1, ctx_idx)                       # (B, Pc) patch-level kpm
    sub_evkpm = torch.gather(pool_evkpm, 1, ctx_idx[..., None].expand(Bc, Pc, K))
    return sub_ev, sub_cen, sub_kpm, sub_evkpm


def run_event_encoder(enc, patch_events, patch_kpm, ev_kpm):
    # Flatten (B, P, K, F) -> (B, P*K, F), run per-event encoder, then
    # mean-pool per patch (with KPM masking) -> (B, P, D).
    B, P, K, Fdim = patch_events.shape
    coord_dim = cfg.coord_dim
    flat = patch_events.view(B, P*K, Fdim)
    coords = flat[..., :coord_dim]
    signal = flat[..., coord_dim:]
    ev_kpm_flat = ev_kpm.view(B, P*K)
    orderings = event_orderings(coords, kpm=ev_kpm_flat)
    g_event = enc(signal, coords, orderings, key_padding_mask=ev_kpm_flat)  # (B, P*K, D)
    # Per-patch mean-pool with masking
    g_view = g_event.view(B, P, K, -1)
    mask = (~ev_kpm).float().unsqueeze(-1)            # (B, P, K, 1)
    g_patch = (g_view * mask).sum(dim=2) / mask.sum(dim=2).clamp(min=1)     # (B, P, D)
    return g_event, g_patch


def quick_probe(num_epochs=cfg.probe_epochs, lr=1e-3, wd=1e-4):
    enc = _unwrap(context_encoder)
    saved_train = enc.training
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    probe = LinearProbe(cfg.d_model, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    def _z(b):
        sub_ev, sub_cen, sub_kpm, sub_evkpm = gather_ctx_subset(b)
        with torch.no_grad():
            enc.eval()
            _, g_patch = run_event_encoder(enc, sub_ev, sub_kpm, sub_evkpm)
        mask = (~sub_kpm).unsqueeze(-1).float()
        return (g_patch * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    for _ in range(num_epochs):
        probe.train()
        for b in train_eval_loader:
            y = b["label"].to(DEVICE)
            z = _z(b)
            opt.zero_grad(set_to_none=True)
            ce(probe(z), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            z = _z(b)
            correct += (probe(z).argmax(-1) == y).sum().item(); total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    enc.train(saved_train)
    return correct / max(total, 1)


## 8. Training loop

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            # Patch tensors
            pool_ev   = batch["pool_patch_events"].to(DEVICE)              # (B, P, K, F)
            pool_kpm  = batch["pool_patch_kpm"].to(DEVICE)
            pool_evkpm= batch["pool_event_kpm"].to(DEVICE)                 # (B, P, K)
            tgt_idx   = batch["tgt_patch_idx"].to(DEVICE)                   # (B, n_tgt)
            tgt_cen   = batch["tgt_patch_centroids"].to(DEVICE)             # (B, n_tgt, 3)

            # ── Context: gather ctx patches → run per-event encoder → patch tokens ──
            sub_ev, sub_cen, sub_kpm, sub_evkpm = gather_ctx_subset(batch)
            _, g_ctx_patch = run_event_encoder(context_encoder,
                                                sub_ev, sub_kpm, sub_evkpm)
                                                                            # (B, Pc, D)

            # ── Target: full per-event encoder → patch features → gather targets ──
            with torch.no_grad():
                _, g_tgt_patch = run_event_encoder(target_encoder,
                                                    pool_ev, pool_kpm, pool_evkpm)
                                                                            # (B, P, D)
                Bg, Pg, Dg = g_tgt_patch.shape
                h_tgt_raw = torch.gather(g_tgt_patch, 1,
                                          tgt_idx.unsqueeze(-1).expand(Bg, tgt_idx.size(1), Dg))
                if cfg.use_centering:
                    target_center.update(h_tgt_raw)
                    h_tgt = F.layer_norm(target_center(h_tgt_raw), (h_tgt_raw.size(-1),))
                else:
                    h_tgt = F.layer_norm(h_tgt_raw, (h_tgt_raw.size(-1),))

            # ── Predictor (perceiver, n_tgt queries → context patches) ──
            h_pred = predictor(g_ctx_patch, tgt_cen,
                                ctx_coords=sub_cen if cfg.predictor_pos_symmetric else None,
                                ctx_key_padding_mask=sub_kpm)                # (B, n_tgt, D)

            loss = jepa_loss(h_pred, h_tgt, loss_type=cfg.loss_type)
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                d = diag_dict(loss, h_pred, h_tgt, g_ctx, target_center)
                print(fmt_diag(d, global_step, epoch, scheduler.get_last_lr()[0], m))
                history["diag_steps"].append(global_step); history["diag_log"].append(d)

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg

        ckpt_state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss,
            "best_probe_acc": best_probe_acc, "probe_no_improve": probe_no_improve,
            "history": history,
        }
        if improved: save_atomic(ckpt_state, PBB_BEST)
        save_atomic(ckpt_state, PBB_LAST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  avg_loss={avg:.4f}  m={m:.4f}  {time.time()-t0:.1f}s"
              + ("  ★ new best" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_probe = time.time()
            print(f"  → probe at epoch {epoch}...")
            acc = quick_probe()
            history.setdefault("probe_accs", []).append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_probe:.1f}s)")
            if acc > best_probe_acc:
                best_probe_acc = acc; probe_no_improve = 0
                ckpt_state["best_probe_acc"] = best_probe_acc
                ckpt_state["probe_no_improve"] = probe_no_improve
                ckpt_state["history"] = history
                save_atomic(ckpt_state, PBB_PROBE_BEST)
                print(f"  ★ new best probe acc → {best_probe_acc:.4f}")
            else:
                probe_no_improve += 1
                print(f"  no probe improvement ({probe_no_improve}/{cfg.probe_patience})")
            if cfg.probe_patience > 0 and probe_no_improve >= cfg.probe_patience:
                print(f"\nEarly stopping at epoch {epoch}: best probe acc = {best_probe_acc:.4f}")
                break

    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 9. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("avg loss"); axes[0].set_title("JEPA loss"); axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    cos_mean = [d["cos_mean"] for d in history["diag_log"]]
    axes[1].plot(steps, cos_mean, color='C2'); axes[1].set_ylim(-0.1, 1.05)
    axes[1].set_xlabel("step"); axes[1].set_title("cos(h_pred, h_tgt)"); axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3')
    axes[2].set_ylim(0, 100); axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe acc (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()
